# Anticipation in Football: From Predicting Opponents to Understanding Teammates
### MIT Sloan Sports Analytics Conference (SSAC 2027) Research Abstract Reproduction Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fady-nasser/anticipation/blob/main/notebooks/reproduce_anticipation.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repository-blue.svg)](https://github.com/fady-nasser/anticipation)
[![Code license: MIT](https://img.shields.io/badge/Code%20license-MIT-blue.svg)](https://github.com/fady-nasser/anticipation/blob/main/LICENSE)

This notebook reproduces the complete analysis, anticipation event counts, and figures presented in the SSAC 2027 submission.
* **No raw tracking coordinates or proprietary archives required.**
* Uses the included 5 Hz extracted velocity profiles ($v_x, v_y$) and continuous ball dynamics.
* Full execution from data download to publication-quality figure rendering takes **< 30 seconds**.


## 1. Environment Setup & Data Acquisition
Clone the repository to get the pre-extracted velocity features and reproduction scripts, or download directly via `curl` / `gdown`.


In [ ]:
# Clone repository if running in Google Colab
import os, sys

if not os.path.exists('data/extracted_features'):
    print('Cloning repository from GitHub...')
    !git clone https://github.com/fady-nasser/anticipation.git
    %cd anticipation

# Install minimal dependencies
!pip install -q numpy pandas scipy matplotlib seaborn
print('Environment ready!')


## 2. Imports and Configuration


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.linalg import lstsq

print('Numpy version:', np.__version__)
print('Pandas version:', pd.__version__)


## 3. Inspecting the Extracted Feature Data
To comply with proprietary data restrictions, we share **5 Hz directional velocity profiles** ($v_x, v_y$) rather than raw pitch coordinates ($x, y$). Let's inspect the extracted features for the 2022 World Cup Final:


In [ ]:
# Load Match 10517 (World Cup Final: Argentina vs France)
data_final = np.load('data/extracted_features/match_10517_final_features.npz')
print('Total feature arrays in file:', len(data_final.files))
print('Sample array names:', data_final.files[:12])

# Display a preview table of velocity features
preview_df = pd.DataFrame({
    'Period': data_final['period'][:10],
    'Time (s)': np.round(data_final['t'][:10], 2),
    'Ball vx (m/s)': np.round(data_final['ball_vx'][:10], 2),
    'Ball vy (m/s)': np.round(data_final['ball_vy'][:10], 2),
    'Messi vx (ARG_10)': np.round(data_final['ARG_10_vx'][:10], 2),
    'Messi vy (ARG_10)': np.round(data_final['ARG_10_vy'][:10], 2),
    'Mbappe vx (FRA_10)': np.round(data_final['FRA_10_vx'][:10], 2),
    'Mbappe vy (FRA_10)': np.round(data_final['FRA_10_vy'][:10], 2),
})
preview_df


## 4. Granger Causality Conditioning on Continuous Ball Motion
In each rolling 15-second tactical window (75 frames at 5 Hz), we test whether Player A's velocity history significantly explains Player B's velocity 1 to 2 seconds later ($F > 3.14$, $p < 0.05$), after controlling for Player B's own momentum and continuous ball dynamics.


In [ ]:
def zscore(arr):
    m = np.nanmean(arr)
    s = np.nanstd(arr)
    return np.zeros_like(arr) if s < 1e-9 else (arr - m) / s

def ols_rss(Y, X):
    mask = ~(np.isnan(Y) | np.any(np.isnan(X), axis=1))
    if mask.sum() < 15:
        return np.nan, 0
    Ym, Xm = Y[mask], X[mask]
    b, _, _, _ = lstsq(np.c_[np.ones(len(Ym)), Xm], Ym, rcond=None)
    r = Ym - np.c_[np.ones(len(Ym)), Xm] @ b
    return np.sum(r**2), mask.sum()

def test_window_granger(vxA, vyA, vxB, vyB, bx, lag=2, f_critical=3.14):
    n = len(vxA)
    if n <= lag + 5: return False
    if np.nanstd(vxA) < 0.2 and np.nanstd(vyA) < 0.2: return False
    if np.nanstd(vxB) < 0.2 and np.nanstd(vyB) < 0.2: return False
    
    def make_lags(a, l_list):
        cols = []
        for l in l_list:
            c = np.full(n, np.nan); c[l:] = a[:n-l]; cols.append(c)
        return np.column_stack(cols)
    
    Y = zscore(vyB)
    ol = list(range(1, lag + 1))
    X_r = np.c_[make_lags(zscore(vxB), ol), make_lags(zscore(vyB), ol), make_lags(zscore(bx), ol)]
    X_u = np.c_[X_r, make_lags(zscore(vxA), [lag]), make_lags(zscore(vyA), [lag])]
    
    rss_r, nr = ols_rss(Y, X_r)
    rss_u, nu = ols_rss(Y, X_u)
    if np.isnan(rss_r) or np.isnan(rss_u) or nu < 20 or rss_u < 1e-12: return False
    
    dfn = X_u.shape[1] - X_r.shape[1]
    dfd = nu - X_u.shape[1] - 1
    if dfd < 5 or dfn < 1: return False
    return ((rss_r - rss_u) / dfn) / (rss_u / dfd) > f_critical

print('Granger Engine Defined!')


## 5. Figure 1: Within-Team Anticipation Maps
Reproducing cumulative within-team anticipation event counts for **France**, **Argentina**, **Morocco**, and **Serbia**.


In [ ]:
# Load within-team count matrices
fra_counts = pd.read_csv('data/count_matrices/france_anticipation_counts.csv', index_col=0)
arg_counts = pd.read_csv('data/count_matrices/argentina_anticipation_counts.csv', index_col=0)
mar_counts = pd.read_csv('data/count_matrices/morocco_anticipation_counts.csv', index_col=0)
srb_counts = pd.read_csv('data/count_matrices/serbia_anticipation_counts.csv', index_col=0)

for df in [fra_counts, arg_counts, mar_counts, srb_counts]:
    np.fill_diagonal(df.values, 0)

fig, axes = plt.subplots(2, 2, figsize=(22, 20))
fig.suptitle('FIFA World Cup 2022 : Cumulative Anticipation Event Counts (15s Windows)\n'
             'Total Discrete Tactical Phases Where Leader Triggered Follower (Controlling for Ball Motion)',
             fontsize=16, fontweight='bold', y=0.98)

teams = [
    ('(A) France (2022 WC Final vs Argentina)', fra_counts),
    ('(B) Argentina (2022 WC Semi-Final 3-0 vs Croatia)', arg_counts),
    ('(C) Morocco (2022 WC Semi-Final vs France)', mar_counts),
    ('(D) Serbia (2022 WC Group Stage 3-3 vs Cameroon)', srb_counts),
]
positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

for idx, (title, cdf) in enumerate(teams):
    r, c = positions[idx]
    ax = axes[r, c]
    sns.heatmap(cdf, ax=ax, cmap='Blues', annot=True, fmt='d', linewidths=0.4,
                mask=None, cbar_kws={'label': 'Anticipation Event Count'}, vmin=0)
    flat = cdf.stack()
    top_pair = flat.idxmax()
    top_val = flat.max()
    ax.set_title(f"{title}\nTop Pair: {top_pair[0]} -> {top_pair[1]} ({top_val} times)", fontsize=12, fontweight='bold')
    ax.set_xlabel('Follower (Reactor)', fontsize=10)
    ax.set_ylabel('Leader (Trigger)', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 6. Figure 2: Intra-Team vs. Cross-Team Anticipation Networks
Reproducing the 2022 World Cup Final network: **(A) France In-Team**, **(B) Argentina Reads France**, and **(C) France Reads Argentina**.


In [ ]:
# Load cross-team count matrices
arg_fra = pd.read_csv('data/count_matrices/cross_arg_leads_fra_counts.csv', index_col=0)
fra_arg = pd.read_csv('data/count_matrices/cross_fra_leads_arg_counts.csv', index_col=0)

fig, axes = plt.subplots(1, 3, figsize=(28, 9.5))
fig.suptitle('FIFA World Cup Final 2022 : In-Team vs. Cross-Team Anticipation Event Counts (15s Windows)\n'
             'Total Discrete Moments Leader Triggered Follower (Controlling for Ball Motion, No Masks)',
             fontsize=16, fontweight='bold', y=0.98)

# Panel A: France In-Team
sns.heatmap(fra_counts, ax=axes[0], cmap='YlGnBu', annot=True, fmt='d', linewidths=0.3,
            mask=None, cbar_kws={'label': 'Event Count (15s Windows)'}, vmin=0)
axes[0].set_title('(A) France In-Team Chemistry\n(Leader: France -> Follower: France)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Follower (France)', fontsize=10); axes[0].set_ylabel('Leader (France)', fontsize=10)
axes[0].tick_params(axis='x', rotation=45); axes[0].tick_params(axis='y', rotation=0)

# Panel B: Argentina Reads France
sns.heatmap(arg_fra, ax=axes[1], cmap='YlGnBu', annot=True, fmt='d', linewidths=0.3,
            mask=None, cbar_kws={'label': 'Event Count (15s Windows)'}, vmin=0)
axes[1].set_title('(B) Cross-Team: Argentina Reads France\n(Leader: Argentina Defender/Midfielder -> Follower: France Attacker)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Follower (France)', fontsize=10); axes[1].set_ylabel('Leader (Argentina)', fontsize=10)
axes[1].tick_params(axis='x', rotation=45); axes[1].tick_params(axis='y', rotation=0)

# Panel C: France Reads Argentina
sns.heatmap(fra_arg, ax=axes[2], cmap='YlGnBu', annot=True, fmt='d', linewidths=0.3,
            mask=None, cbar_kws={'label': 'Event Count (15s Windows)'}, vmin=0)
axes[2].set_title('(C) Cross-Team: France Reads Argentina\n(Leader: France Defender/Midfielder -> Follower: Argentina Attacker)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Follower (Argentina)', fontsize=10); axes[2].set_ylabel('Leader (France)', fontsize=10)
axes[2].tick_params(axis='x', rotation=45); axes[2].tick_params(axis='y', rotation=0)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


## 7. Key Findings Summary
* **France:** Central double-pivot synchronization between Tchouaméni and Rabiot (**293 / 275 phases**).
* **Argentina:** Mac Allister and Messi primary playmaking axis (**218 phases**); Álvarez opening attacking pockets for Messi (**169 phases**).
* **Morocco:** Near-perfect bilateral wing synergy between Ziyech and Hakimi (**187 / 183 phases**).
* **Serbia:** Playmaker Tadić triggering box runs for Mitrović (**215 phases**).
* **World Cup Final Cross-Team:** Mac Allister reading Tchouaméni (**320 phases**); Tchouaméni tracking Mac Allister (**308 phases**), Messi (**287 phases**), and Álvarez (**274 phases**).
